# Diagnostic Reasoning Assistant (DDXPlus) — Notebook 03: RAG & Explainability

**Grounded explanations for Top‑k diagnoses + next-question selection (TF‑IDF baseline), integrated with the saved token LR + IG policy**

**Author:** Julia Parnis   
**Date:** March 1, 2026  

## Project Overview

**Project goal:** Build an AI-powered diagnostic assistant that provides a *ranked differential* through iterative questioning, with a focus on improving detection of rare / overlooked conditions.

**Scope of this notebook:** Integrate a Retrieval-Augmented Generation (RAG) layer on top of the trained token-level Logistic Regression model to enable:
- Natural-language question phrasing (ask → patient answers → re-rank)
- Evidence-backed explainability (why this disease? what does the evidence mean?)
- Cited differential diagnosis summaries grounded in the DDXPlus knowledge base

This notebook focuses on *retrieval pipeline design, prompt engineering, and end-to-end session loop validation*.

**Planned (later phases):**
- Streamlit / API deployment (interactive web interface)
- Advanced question selection policies (reinforcement learning / active learning)
- Evaluation of RAG output quality (faithfulness, citation accuracy, clinical coherence)

**Inputs from Notebook 02:**
- `logreg_token_calibrated.joblib` — calibrated token model for probability ranking
- `preprocessors_token.joblib` — mlb_token, ohe, scaler, label_encoder
- `policy_artifacts.joblib` — p_e_given_d, evidence_bases, evidence_index
- `.py` modules in `assistant/` — types, encoding, inference, policy, artifacts

**Dataset:** DDXPlus (synthetic) — ~1.3M patient cases, 49 pathologies, evidence-based features  
**Source:** Hugging Face: `aai530-group6/ddxplus`  
**Primary reference:** Fansi Tchango et al. (2022)

> **Important:** DDXPlus is synthetic (generated from medical knowledge bases + an AD system).  
> This notebook is for research/education only; it does **not** produce a deployable clinical tool.

## Notebook 03 objective: RAG-powered explainability and iterative questioning

**Objective:** Build and validate a Retrieval-Augmented Generation layer on top of the trained token-level ML model to enable a natural-language diagnostic assistant, emphasising:
- Grounded question phrasing (questions generated from retrieved evidence descriptions, not hardcoded)
- Evidence-backed explainability (cited reasoning for the ranked differential)
- End-to-end session loop (ask → patient answers → re-rank → next question)
- Faithful retrieval (retrieved context is relevant, non-hallucinated, and traceable to source)

**Success criteria (for this notebook):**
- RAG pipeline runs end-to-end from a cold `DiagnosisSession` to a cited differential summary
- Questions are phrased in natural language and grounded in `release_evidences.json`
- Top-k differential is explained with at least one retrieved supporting fact per disease
- Session loop completes at least 5 question–answer turns without errors
- All prompts, retrievals, and outputs are logged and saved to `outputs/rag/`


# Notebook 03 — RAG Explanations for the DDXPlus Diagnostic Reasoning Assistant

## What we are building
We are adding a **local Retrieval-Augmented Generation (RAG)** layer that produces:
1) grounded explanations for **why the current Top‑k diagnoses** are ranked highly, and  
2) grounded explanations for **why the next question** was selected by the information‑gain policy.

## What stays fixed (no retraining)
- The ranking engine remains the **Logistic Regression token/value-level model** (972 token vocab; 978 total features with demographics + numeric complexity).
- The question policy remains **base-level (223 evidences)** using the likelihood table `p_e_given_d = P(e=1 | disease)`.
- For policy decisions, we prefer **calibrated probabilities** when available because information gain depends on meaningful uncertainty estimates.

## Data sources for retrieval (local, reproducible)
We build a retrieval corpus from:
- `release_evidences.json` (evidence IDs → question text, data type, default values, value meanings)
- `release_conditions.json` (condition names + symptom/antecedent evidence sets, severity, ICD hints)

Optional: we may add a small curated “mini-manual” layer (short markdown notes per condition) to improve explanation quality without requiring any external web access.

## Safety / scope
This project is **educational only** and uses the **synthetic DDXPlus dataset**. Explanations are phrased as learning support (how evidence patterns relate to diagnoses in this dataset and in general) and are not medical advice.

# Workplan (Notebook 03)

## 0) Setup and reproducibility
- Define paths (`data/`, `models/`, `outputs/rag/`) and create output folders.
- Load saved artifacts (preprocessors, calibrated token LR when available, policy table).
- Confirm feature shapes and that the pipeline runs without retraining.

## 1) Load dataset metadata (the retrieval “ground truth”)
- Load `release_evidences.json` (223 evidence bases).
- Load `release_conditions.json` (49 conditions).
- Build helper functions to format evidence definitions and condition summaries.

## 2) Build the local retrieval corpus (TF‑IDF baseline)
- Create documents for:
  - each evidence base (question text + type + default/value meanings)
  - each condition (name + key symptoms/antecedents listed in the metadata)
- Store document metadata (doc_type, evidence_id / condition_id, fields used).
- Fit a TF‑IDF vectorizer and persist the corpus + vectors to `outputs/rag/`.

## 3) (Optional) Add a curated mini-manual layer (49 conditions)
- Create one short markdown note per condition:
  - “what it is” (1–2 sentences)
  - “typical evidence themes” (based on dataset metadata)
  - “common confusions” (dataset-adjacent, educational framing)
- Include these notes as additional retrievable documents.

## 4) Retrieval API (clean interface used by the loop)
- Implement `retrieve(query, top_n)` returning:
  - ranked docs + similarity scores
  - doc metadata for grounding/provenance
- Add lightweight tests: deterministic results given fixed corpus.

## 5) Grounded explanation templates
- Template A: **“Why these Top‑k diagnoses now?”**
  - Use: posterior Top‑k + retrieved condition summaries + retrieved evidence definitions.
- Template B: **“Why ask this next question?”**
  - Use: chosen evidence base + question text + IG score + how it separates top diagnoses.

## 6) Integrate with the existing assistant loop
- Session state → token inference → posterior → IG policy → chosen question.
- Run retrieval to ground:
  - Top‑k explanation
  - next-question explanation
- Keep outputs consistent with the smoke test expectations (no retraining, same artifacts).

## 7) Demo trace + saved outputs
- Run a short simulated session (a few turns).
- Save:
  - retrieved contexts
  - explanations
  - a compact “trace log” for debugging / demo.

## 0) Setup & reproducibility

This section configures paths, seeds, display settings, and figure defaults (Zoom-friendly) consistent with Notebook 02.  
It also creates the new stage folders: `outputs/rag/` and `figures/rag/`.

In [ ]:
# ============================================================
# 0.1 Imports + project-root check + src import
# ============================================================

from __future__ import annotations

import os
import sys
import random
import json
from pathlib import Path
from dataclasses import dataclass, field
import warnings
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

# Visualization (match Notebook 02)
import matplotlib.pyplot as plt
from cycler import cycler

# Artifacts / ML utilities (needed later in this notebook)
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Silence noisy warnings (keep notebook readable)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
# -------------------------
# Project root detection
# -------------------------
def find_project_root(start: Path | None = None) -> Path:
    """
    Find repo root by walking upward until data/ and models/ are found.
    Works whether the notebook is run from repo root or from notebooks/.
    """
    start = Path.cwd() if start is None else start.resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "models").exists():
            return p
    # Fallback: if running in a minimal environment, treat cwd as root
    return start


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"

print(f"✓ PROJECT_ROOT = {PROJECT_ROOT}")

# Ensure src/ is importable when running from notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

✓ PROJECT_ROOT = c:\Julia\Ironhack\Week20-24_Final_project\diagnostic-reasoning-assistant


In [6]:
# Quick check-up
from src.artifacts import load_json  # noqa: E402
print("✓ src imports OK")

✓ src imports OK


In [5]:
# ============================================================
# 0.2 Reproducibility: seeds + version snapshot
# ============================================================

random_seed = 42

os.environ["PYTHONHASHSEED"] = str(random_seed)
np.random.seed(random_seed)
random.seed(random_seed)

print(f"✓ Random seed set to: {random_seed}")

# Capture a small environment snapshot (helps later when debugging)
import sklearn  # local import to keep import cell tidy

env_info = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}

print("Environment snapshot:")
for k, v in env_info.items():
    print(f"  - {k}: {v}")

✓ Random seed set to: 42
Environment snapshot:
  - python: 3.12.7
  - numpy: 2.4.2
  - pandas: 3.0.0
  - scikit_learn: 1.8.0
  - joblib: 1.5.3


In [8]:
# ============================================================
# 0.3 Config (Notebook-02 style) + create rag folders
# ============================================================

@dataclass
class Config:
    """
    Central configuration for this project.

    Folder strategy:
    - Fixed root folders: outputs/, models/, figures/
    - Subfolders: outputs/{stage}/, figures/{stage}/
    - File naming: stable filenames (no date tagging)
    """

    # ── Data paths ───────────────────────────────────────────
    data_dir: Path            = Path("data")
    evidence_map_path: Path   = Path("data/release_evidences.json")
    conditions_map_path: Path = Path("data/release_conditions.json")

    # ── Output folders ───────────────────────────────────────
    output_dir: Path  = Path("outputs")
    models_dir: Path  = Path("models")
    figures_dir: Path = Path("figures")

    # ── Reproducibility ──────────────────────────────────────
    random_seed: int = random_seed


config = Config()

# Ensure stage folders exist (Notebook 02 created eda/ml; Notebook 03 adds rag)
for stage in ["eda", "ml", "rag"]:
    (config.output_dir / stage).mkdir(parents=True, exist_ok=True)
    (config.figures_dir / stage).mkdir(parents=True, exist_ok=True)

config.models_dir.mkdir(parents=True, exist_ok=True)

print("✓ Directories ready:")
print(f"  - {config.output_dir / 'rag'}")
print(f"  - {config.figures_dir / 'rag'}")
print(f"  - {config.models_dir}")

✓ Directories ready:
  - outputs\rag
  - figures\rag
  - models


In [9]:
# ============================================================
# 0.4 Visualization Settings (same as Notebook 02)
# ============================================================

# Apply clean seaborn style (no seaborn import needed)
plt.style.use("seaborn-v0_8-white")

# IBM Design colorblind-safe palette
IBM_COLORS = {
    "blue": "#648FFF",
    "purple": "#785EF0",
    "magenta": "#DC267F",
    "orange": "#FE6100",
    "yellow": "#FFB000",
    "teal": "#06A39B",
    "gray": "#5F6368",
}

# Figure defaults (presentation-optimized)
plt.rcParams.update({
    # Figure size and resolution
    "figure.figsize": (6, 4),          # Good for 2+ figs per slide
    "figure.dpi": 120,                 # Screen display
    "savefig.dpi": 300,                # High-quality save

    # Font sizes (readable on Zoom)
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,

    # Appearance (subtle, professional)
    "axes.edgecolor": IBM_COLORS["gray"],
    "axes.linewidth": 1.2,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.8,
    "grid.alpha": 0.6,
})

# Set IBM color cycle (for multi-line plots)
plt.rcParams["axes.prop_cycle"] = cycler(color=[
    IBM_COLORS["blue"],
    IBM_COLORS["teal"],
    IBM_COLORS["purple"],
    IBM_COLORS["magenta"],
    IBM_COLORS["orange"],
])

print("✓ Plot style configured (Notebook-02 consistent)")

# -------------------------
# Display settings (Notebook 02 style)
# -------------------------
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 4)

print("✓ Display settings configured")

✓ Plot style configured (Notebook-02 consistent)
✓ Display settings configured


In [10]:
# ============================================================
# 0.5 Save helpers (same as Notebook 02)
# ============================================================

def save_fig(fig, filename: str, stage: str = "ml"):
    """
    Save figure into figures/{stage}/filename.

    stage: "eda" or "ml" or "rag"
    """
    out_dir = config.figures_dir / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"✓ Saved: {path}")


def save_table_csv(df: pd.DataFrame, filename: str, stage: str = "ml", index: bool = False):
    out_dir = config.output_dir / stage
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / filename
    df.to_csv(path, index=index)
    print(f"✓ Saved: {path}")

## Step 1 — Local retrieval corpus (DDXPlus metadata)

We build a **local** corpus from:
- `release_evidences.json` (223 evidence definitions: question text, data type, default value)
- `release_conditions.json` (49 condition entries: ICD10, severity, symptom/antecedent evidence sets)

This baseline corpus enables TF‑IDF retrieval (fast, reproducible) and is the foundation for grounded explanations.

In [11]:
# ============================================================
# 1.1 Load sources (DDXPlus mapping files)
# ============================================================

evidences_map = load_json(config.evidence_map_path)
conditions_map = load_json(config.conditions_map_path)

print(f"✓ Loaded release_evidences.json: {len(evidences_map)} evidences")
print(f"✓ Loaded release_conditions.json: {len(conditions_map)} conditions")

# Basic sanity checks (expected: 223 evidences, 49 conditions)
assert len(evidences_map) == 223, "Unexpected number of evidences (expected 223)."
assert len(conditions_map) == 49, "Unexpected number of conditions (expected 49)."

✓ Loaded release_evidences.json: 223 evidences
✓ Loaded release_conditions.json: 49 conditions


In [14]:
# Looking into evidences_map and conditions_map files

# ---- Example: one condition (lists evidence BASE codes) ----
example_condition = "Acute rhinosinusitis"
cond = conditions_map[example_condition]

print("\nCondition example:", example_condition)
print("  keys:", list(cond.keys()))
print("  icd10-id:", cond.get("icd10-id"))
print("  severity:", cond.get("severity"))
print("  #symptoms:", len(cond.get("symptoms", {})))
print("  #antecedents:", len(cond.get("antecedents", {})))
print("  symptom bases (first 12):", list(cond["symptoms"].keys())[:12])

# ---- Example: one evidence (contains question text + type + default) ----
example_evidence = "E_139"
ev = evidences_map[example_evidence]

print("\nEvidence example:", example_evidence)
print("  keys:", list(ev.keys()))
print("  question_en:", ev.get("question_en"))
print("  data_type:", ev.get("data_type"))
print("  default_value:", ev.get("default_value"))
print("  is_antecedent:", ev.get("is_antecedent"))


Condition example: Acute rhinosinusitis
  keys: ['condition_name', 'cond-name-fr', 'cond-name-eng', 'icd10-id', 'symptoms', 'antecedents', 'severity']
  icd10-id: j01
  severity: 4
  #symptoms: 12
  #antecedents: 10
  symptom bases (first 12): ['E_55', 'E_53', 'E_57', 'E_54', 'E_59', 'E_56', 'E_58', 'E_182', 'E_201', 'E_103', 'E_91', 'E_181']

Evidence example: E_139
  keys: ['name', 'code_question', 'question_fr', 'question_en', 'is_antecedent', 'default_value', 'value_meaning', 'possible-values', 'data_type']
  question_en: Do you have a known heart defect?
  data_type: B
  default_value: 0
  is_antecedent: True


In [12]:
# ============================================================
# 1.2 Build RAG corpus (documents + metadata)
# ============================================================

def _safe_str(x: Any) -> str:
    return "" if x is None else str(x)


def build_evidence_doc(base: str, meta: Dict[str, Any], *, max_value_examples: int = 12) -> Tuple[str, Dict[str, Any]]:
    """
    Build a compact text document for one evidence base.

    We intentionally do NOT dump huge value lists; we include only a small sample of value meanings.
    The full mapping stays available in `evidences_map` for UI rendering later.
    """
    q_en = _safe_str(meta.get("question_en")).strip()
    data_type = _safe_str(meta.get("data_type")).strip()  # B / C / M / ...
    default_value = meta.get("default_value")
    is_antecedent = bool(meta.get("is_antecedent", False))

    value_meaning = meta.get("value_meaning") or {}
    # Pull a small sample of English meanings (skip NA-like where possible)
    examples = []
    for v_code, v_meta in value_meaning.items():
        en = _safe_str((v_meta or {}).get("en")).strip()
        if not en or en.lower() == "na":
            continue
        examples.append(f"{v_code}={en}")
        if len(examples) >= max_value_examples:
            break

    lines = [
        f"Evidence base: {base}",
        f"Question (EN): {q_en}",
        f"Type: {data_type} | Antecedent: {is_antecedent} | Default value: {default_value}",
    ]
    if examples:
        lines.append("Example values: " + "; ".join(examples))

    doc_text = "\n".join([ln for ln in lines if ln.strip()])

    doc_meta = {
        "doc_type": "evidence",
        "base": base,
        "data_type": data_type,
        "is_antecedent": is_antecedent,
    }
    return doc_text, doc_meta


def build_condition_doc(
    condition_name: str,
    meta: Dict[str, Any],
    evidences_map: Dict[str, Any],
    *,
    max_evidence_list: int = 20,
) -> Tuple[str, Dict[str, Any]]:
    """
    Build a retrieval doc for one condition using:
    - ICD10 id
    - severity
    - linked symptom/antecedent evidence bases (with question text)
    """
    icd10 = _safe_str(meta.get("icd10-id")).strip()
    severity = meta.get("severity", None)

    symptoms = list((meta.get("symptoms") or {}).keys())
    antecedents = list((meta.get("antecedents") or {}).keys())

    def q(base: str) -> str:
        return _safe_str((evidences_map.get(base) or {}).get("question_en")).strip()

    symptom_lines = [f"- {b}: {q(b)}" for b in symptoms[:max_evidence_list] if q(b)]
    antecedent_lines = [f"- {b}: {q(b)}" for b in antecedents[:max_evidence_list] if q(b)]

    lines = [
        f"Condition: {condition_name}",
        f"ICD10: {icd10} | Severity: {severity}",
    ]
    if symptom_lines:
        lines.append("Common symptoms in dataset mapping:\n" + "\n".join(symptom_lines))
    if antecedent_lines:
        lines.append("Common antecedents in dataset mapping:\n" + "\n".join(antecedent_lines))

    doc_text = "\n".join(lines)

    doc_meta = {
        "doc_type": "condition",
        "condition_name": condition_name,
        "icd10": icd10,
        "severity": severity,
    }
    return doc_text, doc_meta


# ---- Build corpus ----
rag_docs: List[str] = []
rag_meta: List[Dict[str, Any]] = []

# 1) Evidence docs (223)
for base, meta in evidences_map.items():
    doc_text, doc_meta = build_evidence_doc(base, meta)
    rag_docs.append(doc_text)
    rag_meta.append(doc_meta)

# 2) Condition docs (49) — keys are already condition names
for condition_name, meta in conditions_map.items():
    doc_text, doc_meta = build_condition_doc(condition_name, meta, evidences_map)
    rag_docs.append(doc_text)
    rag_meta.append(doc_meta)

print(f"✓ RAG corpus built: {len(rag_docs)} documents")

# ---- Save a readable copy (JSONL) for inspection ----
corpus_path = config.output_dir / "rag" / "rag_corpus.jsonl"
with corpus_path.open("w", encoding="utf-8") as f:
    for text, meta in zip(rag_docs, rag_meta):
        rec = {"text": text, "meta": meta}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✓ Saved corpus: {corpus_path}")

✓ RAG corpus built: 272 documents
✓ Saved corpus: outputs\rag\rag_corpus.jsonl


In [13]:
# ============================================================
# 1.3 TF-IDF retriever (baseline) + save artifacts + validation
# ============================================================

@dataclass
class LocalTfidfRetriever:
    vectorizer: TfidfVectorizer
    X: Any  # sparse matrix
    docs: List[str]
    meta: List[Dict[str, Any]]

    def search(self, query: str, k: int = 5, *, doc_type: Optional[str] = None) -> List[Dict[str, Any]]:
        """
        Return top-k docs by cosine similarity.
        If doc_type is set ("evidence" or "condition"), filter results to that type.
        """
        qv = self.vectorizer.transform([query])
        sims = cosine_similarity(qv, self.X).ravel()

        order = np.argsort(sims)[::-1]
        results = []
        for idx in order:
            if doc_type is not None and self.meta[idx].get("doc_type") != doc_type:
                continue
            results.append({
                "score": float(sims[idx]),
                "meta": self.meta[idx],
                "text": self.docs[idx],
            })
            if len(results) >= k:
                break
        return results


# Fit TF-IDF (deterministic)
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=8000,
    ngram_range=(1, 2),
)

X = tfidf.fit_transform(rag_docs)

retriever = LocalTfidfRetriever(
    vectorizer=tfidf,
    X=X,
    docs=rag_docs,
    meta=rag_meta,
)

print(f"✓ TF-IDF index built: X shape = {X.shape}")

# Save one compact artifact for later (notebook/app load)
rag_artifacts_path = config.models_dir / "rag_tfidf_artifacts.joblib"
joblib.dump(
    {"vectorizer": tfidf, "X": X, "docs": rag_docs, "meta": rag_meta},
    rag_artifacts_path,
)
print(f"✓ Saved TF-IDF retrieval artifacts: {rag_artifacts_path}")

# --- Quick check-up tests ---
test_queries = [
    "pneumothorax",
    "rash color",
    "chest pain",
]

for q in test_queries:
    hits = retriever.search(q, k=3)
    print("\nQuery:", q)
    for h in hits:
        print("  ", h["meta"], f"score={h['score']:.3f}")

✓ TF-IDF index built: X shape = (272, 3495)
✓ Saved TF-IDF retrieval artifacts: models\rag_tfidf_artifacts.joblib

Query: pneumothorax
   {'doc_type': 'evidence', 'base': 'E_165', 'data_type': 'B', 'is_antecedent': True} score=0.314
   {'doc_type': 'evidence', 'base': 'E_21', 'data_type': 'B', 'is_antecedent': True} score=0.313
   {'doc_type': 'condition', 'condition_name': 'Spontaneous pneumothorax', 'icd10': 'J93', 'severity': 2} score=0.263

Query: rash color
   {'doc_type': 'evidence', 'base': 'E_132', 'data_type': 'C', 'is_antecedent': False} score=0.182
   {'doc_type': 'evidence', 'base': 'E_130', 'data_type': 'C', 'is_antecedent': False} score=0.180
   {'doc_type': 'condition', 'condition_name': 'Influenza', 'icd10': 'j11.1', 'severity': 3} score=0.171

Query: chest pain
   {'doc_type': 'evidence', 'base': 'E_14', 'data_type': 'B', 'is_antecedent': False} score=0.401
   {'doc_type': 'evidence', 'base': 'E_105', 'data_type': 'B', 'is_antecedent': True} score=0.367
   {'doc_type':

## Step 2 — Curated mini‑manual docs (49 conditions)

### Why this helps
Our current condition docs are already grounded, but they are mostly “lists.”
A mini‑manual gives the retriever richer, more structured text, which usually produces more professional explanations in:
- “Why these diagnoses?” (tie the posterior to plausible symptoms/antecedents)
- “Why this question next?” (tie IG selection to what distinguishes top conditions)

### What we will generate (grounded, local-only)
For each condition in `release_conditions.json`, we will write one markdown file containing:
- Condition name (+ ICD‑10 when present)
- DDXPlus severity value (dataset metadata)
- A “Symptoms” section listing the linked evidence questions
- An “Antecedents / risk-factor questions” section listing linked evidence questions
- A short educational disclaimer

These files will be saved to: `outputs/rag/mini_manual/` and will be added to the TF‑IDF corpus.

In [15]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Any, Dict, List, Tuple


def slugify(text: str, max_len: int = 80) -> str:
    """Make a filesystem-friendly slug from a condition name."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"-{2,}", "-", text).strip("-")
    return text[:max_len] if len(text) > max_len else text


def sort_evidence_codes(codes: List[str]) -> List[str]:
    """Sort evidence codes like E_2, E_10, E_100 numerically when possible."""
    def _key(code: str) -> int:
        m = re.search(r"E_(\d+)", code)
        return int(m.group(1)) if m else 10**9
    return sorted(codes, key=_key)


def evidence_bullet(base: str, evidences_map: Dict[str, Any]) -> str:
    """
    Build one bullet line for an evidence base code using release_evidences.json.
    We keep the base code in the text so retrieval can match queries like 'E_55'.
    """
    meta = evidences_map.get(base, {})
    q_en = meta.get("question_en") or base
    data_type = meta.get("data_type")
    if data_type:
        return f"- **{base}** ({data_type}): {q_en}"
    return f"- **{base}**: {q_en}"


def bullets_for_codes(
    codes: List[str],
    evidences_map: Dict[str, Any],
    *,
    max_items: int = 15,
) -> str:
    """Render a (possibly truncated) bullet list for a set of evidence base codes."""
    codes_sorted = sort_evidence_codes(list(codes))
    shown = codes_sorted[:max_items]
    lines = [evidence_bullet(c, evidences_map) for c in shown]

    if len(codes_sorted) > max_items:
        lines.append(f"- … ({len(codes_sorted) - max_items} more omitted for brevity)")
    return "\n".join(lines)


def build_condition_manual_doc(
    condition_name: str,
    meta: Dict[str, Any],
    evidences_map: Dict[str, Any],
    *,
    max_symptoms: int = 15,
    max_antecedents: int = 15,
) -> Tuple[str, Dict[str, Any]]:
    """
    Create a markdown mini-manual doc for one condition using release_conditions.json.
    This is *dataset-grounded metadata* intended for educational explanations.
    """
    icd10 = meta.get("icd10-id", "")
    severity = meta.get("severity", None)

    symptom_codes = list((meta.get("symptoms") or {}).keys())
    antecedent_codes = list((meta.get("antecedents") or {}).keys())

    md = []
    md.append(f"# {condition_name}")
    md.append("")
    md.append("**Educational note:** This text is generated from DDXPlus synthetic metadata. "
              "It is not medical advice and is used only to explain model behavior.")
    md.append("")
    md.append("## Dataset metadata")
    if icd10:
        md.append(f"- ICD‑10: `{icd10}`")
    if severity is not None:
        md.append(f"- DDXPlus severity (dataset field): `{severity}`")
    md.append("")
    md.append("## Symptoms linked to this condition (DDXPlus mapping)")
    md.append(bullets_for_codes(symptom_codes, evidences_map, max_items=max_symptoms) or "- (none listed)")
    md.append("")
    md.append("## Antecedents / risk-factor questions linked to this condition (DDXPlus mapping)")
    md.append(bullets_for_codes(antecedent_codes, evidences_map, max_items=max_antecedents) or "- (none listed)")
    md.append("")
    md.append("## Notes for the assistant")
    md.append("- These are **base evidence codes** (question concepts). The predictor still uses token/value encoding.")
    md.append("- The question policy selects among base codes using information gain; RAG only provides wording/explanations.")

    doc_text = "\n".join(md)

    doc_meta = {
        "type": "manual",
        "condition_name": condition_name,
        "icd10": icd10,
        "severity": severity,
        "n_symptoms": len(symptom_codes),
        "n_antecedents": len(antecedent_codes),
    }
    return doc_text, doc_meta

In [16]:
import json
import pandas as pd

# --- Preconditions: these should already exist from Steps 0–1 ---
assert "config" in globals(), "Expected 'config' from setup cells."
assert "conditions_map" in globals(), "Expected 'conditions_map' loaded from release_conditions.json."
assert "evidences_map" in globals(), "Expected 'evidences_map' loaded from release_evidences.json."

# --- Output folder following the project folder strategy ---
manual_dir = config.output_dir / "rag" / "mini_manual"
manual_dir.mkdir(parents=True, exist_ok=True)

rows = []
index = {}

# Stable ordering for reproducibility
for condition_name in sorted(conditions_map.keys()):
    meta = conditions_map[condition_name]
    doc_text, doc_meta = build_condition_manual_doc(
        condition_name,
        meta,
        evidences_map,
        max_symptoms=18,
        max_antecedents=18,
    )

    fname = f"{slugify(condition_name)}.md"
    path = manual_dir / fname
    path.write_text(doc_text, encoding="utf-8")

    index[condition_name] = fname
    rows.append({
        "condition_name": condition_name,
        "icd10": doc_meta.get("icd10"),
        "severity": doc_meta.get("severity"),
        "n_symptoms": doc_meta.get("n_symptoms"),
        "n_antecedents": doc_meta.get("n_antecedents"),
        "manual_file": fname,
    })

manual_index_path = manual_dir / "index.json"
manual_index_path.write_text(json.dumps(index, indent=2), encoding="utf-8")

manual_summary = pd.DataFrame(rows).sort_values(["severity", "condition_name"], ascending=[True, True])
save_table_csv(manual_summary, "mini_manual_summary.csv", stage="rag", index=False)

print(f"✓ Wrote {len(rows)} mini-manual files to: {manual_dir}")
print(f"✓ Wrote manual index to: {manual_index_path}")
print("✓ Saved summary CSV to outputs/rag/mini_manual_summary.csv")
manual_summary.head(8)

✓ Saved: outputs\rag\mini_manual_summary.csv
✓ Wrote 49 mini-manual files to: outputs\rag\mini_manual
✓ Wrote manual index to: outputs\rag\mini_manual\index.json
✓ Saved summary CSV to outputs/rag/mini_manual_summary.csv


,condition_name,icd10,severity,n_symptoms,n_antecedents,manual_file
4,Acute pulmonary edema,J81.0,1,15,6,acute-pulmonary-edema.md
7,Anaphylaxis,T78.0,1,25,3,anaphylaxis.md
19,Ebola,a98.4,1,10,3,ebola.md
26,Larygospasm,J38.5,1,1,6,larygospasm.md
35,Possible NSTEMI / STEMI,I21,1,12,11,possible-nstemi-stemi.md
1,Acute dystonic reactions,G24.02,2,8,4,acute-dystonic-reactions.md
10,Boerhaave,K22.3,2,11,2,boerhaave.md
18,Croup,J05.0,2,7,3,croup.md


In [17]:


# --- Preconditions: doc builders should already exist from Steps 1–2 ---
assert "build_condition_doc" in globals(), "Expected 'build_condition_doc' from earlier cells."
assert "build_evidence_doc" in globals(), "Expected 'build_evidence_doc' from earlier cells."

# 1) Base corpus: evidences + conditions
docs: List[str] = []
meta: List[Dict[str, Any]] = []

for base in sorted(evidences_map.keys(), key=lambda x: int(x.split("_")[1])):
    text, m = build_evidence_doc(base, evidences_map[base])
    docs.append(text)
    meta.append(m)

for condition_name in sorted(conditions_map.keys()):
    text, m = build_condition_doc(condition_name, conditions_map[condition_name], evidences_map)
    docs.append(text)
    meta.append(m)

# 2) Add mini-manual docs
index = json.loads((manual_dir / "index.json").read_text(encoding="utf-8"))
for condition_name in sorted(index.keys()):
    f = index[condition_name]
    p = manual_dir / f
    docs.append(p.read_text(encoding="utf-8"))
    meta.append({"type": "manual", "condition_name": condition_name, "path": str(p)})

print(f"Corpus docs: {len(docs)} (evidence + condition + manual)")

# 3) Fit TF‑IDF retriever (baseline)
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=6000,
    ngram_range=(1, 2),
)
X_docs = vectorizer.fit_transform(docs)

# 4) Save artifact for later integration with the assistant loop
retriever_artifact = {
    "vectorizer": vectorizer,
    "X_docs": X_docs,
    "docs": docs,
    "meta": meta,
}
retriever_path = config.output_dir / "rag" / "tfidf_retriever.joblib"
joblib.dump(retriever_artifact, retriever_path)

print(f"✓ Saved TF‑IDF retriever artifact to: {retriever_path}")

# 5) Quick QC check query (should return some manual/condition/evidence docs)
test_query = "Acute rhinosinusitis"
qv = vectorizer.transform([test_query])
sims = cosine_similarity(qv, X_docs).ravel()
top_idx = sims.argsort()[::-1][:5]

print(f"\nTop results for query='{test_query}':")
for i in top_idx:
    print(f"  score={float(sims[i]):.4f}  meta={meta[i]}")

Corpus docs: 321 (evidence + condition + manual)
✓ Saved TF‑IDF retriever artifact to: outputs\rag\tfidf_retriever.joblib

Top results for query='Acute rhinosinusitis':
  score=0.1504  meta={'doc_type': 'condition', 'condition_name': 'Acute rhinosinusitis', 'icd10': 'j01', 'severity': 4}
  score=0.1345  meta={'type': 'manual', 'condition_name': 'Acute rhinosinusitis', 'path': 'outputs\\rag\\mini_manual\\acute-rhinosinusitis.md'}
  score=0.0551  meta={'doc_type': 'condition', 'condition_name': 'Chronic rhinosinusitis', 'icd10': 'j32', 'severity': 5}
  score=0.0490  meta={'type': 'manual', 'condition_name': 'Chronic rhinosinusitis', 'path': 'outputs\\rag\\mini_manual\\chronic-rhinosinusitis.md'}
  score=0.0412  meta={'doc_type': 'condition', 'condition_name': 'Acute laryngitis', 'icd10': 'J04.0', 'severity': 4}


## Step 3 — RAG explanation building blocks (templates + grounded snippets)

We now connect the pieces into the interactive flow:

**session → token inference → posterior → IG policy → chosen question → RAG explanations**

We will implement two grounded templates:
1) **Why these Top‑k diagnoses now?**
2) **Why ask this next question?**

RAG stays strictly as a *wording/explanation* layer: it does not modify model probabilities.

In [18]:
# ============================================================
# 3.1 Load artifacts (no retraining) + import src utilities
# ============================================================

from pathlib import Path
import joblib
import numpy as np

from src.artifacts import load_json, load_preprocessors, resolve_best_token_model, load_policy_artifacts
from src.types import DiagnosisSession, FeatureSpec
from src.inference import predict_topk_from_session_token
from src.policy import choose_next_question_info_gain

# Paths (project-root relative)
MODELS_DIR = Path("models")
DATA_DIR = Path("data")
OUT_RAG = Path("outputs") / "rag"

# Mapping files (for question text and condition names)
evidences_map = load_json(DATA_DIR / "release_evidences.json")
conditions_map = load_json(DATA_DIR / "release_conditions.json")

# Preprocessors (token)
# NOTE: This path is the expected artifact name in the project state/smoke test.
pre = load_preprocessors(MODELS_DIR / "preprocessors_token.joblib")
mlb_token = pre["mlb_token"]
ohe = pre["ohe"]
scaler = pre["scaler"]
label_encoder = pre["label_encoder"]
CAT_COLS = list(pre["CAT_COLS"])
NUM_COLS = list(pre["NUM_COLS"])

# Token model (prefer calibrated when present)
model_token = resolve_best_token_model(MODELS_DIR)

# Policy artifacts
pol = load_policy_artifacts(MODELS_DIR / "policy_artifacts.joblib")
p_e_given_d = pol["p_e_given_d"]
evidence_bases = list(pol["evidence_bases"])

# FeatureSpec for inference (kept explicit to avoid mismatch)
feature_spec = FeatureSpec(
    evidence_mode="token",
    cat_cols=CAT_COLS,
    num_cols=NUM_COLS,
    age_group_func=None,
)

# Retriever artifact (built in previous steps)
retriever_path = OUT_RAG / "tfidf_retriever.joblib"
retr = joblib.load(retriever_path)

print("✓ Loaded artifacts:")
print("  - mlb_token vocab:", len(mlb_token.classes_))
print("  - model_token:", type(model_token).__name__)
print("  - policy p_e_given_d:", np.array(p_e_given_d).shape, "with", len(evidence_bases), "bases")
print("  - retriever:", retriever_path)

✓ Loaded artifacts:
  - mlb_token vocab: 972
  - model_token: CalibratedClassifierCV
  - policy p_e_given_d: (49, 223) with 223 bases
  - retriever: outputs\rag\tfidf_retriever.joblib


In [19]:
# ============================================================
# 3.2 Retrieval helper (TF‑IDF artifact)
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

def retrieve_snippets(query: str, *, k: int = 4, doc_type: str | None = None):
    """
    Retrieve top-k snippets from the TF‑IDF corpus.

    doc_type can be:
      - None (no filter)
      - 'manual' (condition mini-manual docs)
      - 'condition' or 'evidence' (from earlier corpus builder)
    """
    vectorizer = retr["vectorizer"]
    X_docs = retr["X_docs"]
    docs = retr["docs"]
    meta = retr["meta"]

    qv = vectorizer.transform([query])
    sims = cosine_similarity(qv, X_docs).ravel()
    order = sims.argsort()[::-1]

    hits = []
    for idx in order:
        m = meta[idx]
        if doc_type is not None and m.get("type") != doc_type and m.get("doc_type") != doc_type:
            continue
        hits.append({"score": float(sims[idx]), "meta": m, "text": docs[idx]})
        if len(hits) >= k:
            break
    return hits

### 3.3 User-facing explanations (plain language, medically grounded)

This notebook uses a **two-part approach**:
1) a statistical ranking engine produces a short list of likely diagnoses based on the information recorded so far, and  
2) a retrieval layer provides **human-readable wording** and **grounded explanations** using local documents (DDXPlus mappings + optional mini‑manual notes).

**Audience & tone (important):**
- Explanations are written for medical students or non-technical learners.
- Technical ML terms are avoided (e.g., “token model”, “logistic regression”, “information gain”).
- Wording uses *we* voice or passive voice (avoid “you”), and keeps a calm, educational tone.

**Grounding rules (no hallucinations):**
- The assistant only explains using:
  - recorded findings already present in the session, and
  - retrieved context from the local corpus built from DDXPlus mapping files, plus
  - optional short “mini‑manual” summaries that we curate (1–2 sentences per condition) with references.
- If a clinical fact is not present in the curated mini‑manual (or the retrieved corpus), it is not stated as a fact.

**What the assistant explains (two templates):**

**A) “Why these Top‑k diagnoses now?”**
- A short list of the current leading diagnoses (Top‑k).
- For each diagnosis:
  - 1 short “what this condition is” line (from mini‑manual if available; otherwise a minimal label),
  - 2–3 “why it is on the list now” points tied to the recorded findings so far,
  - optionally: 1 sentence on what additional information would help separate it from nearby alternatives.

**B) “Why ask this next question?”**
- The next question is chosen because it is expected to **separate the leading possibilities quickly**.
- Explanation is phrased clinically:
  - “This question helps because the answer often differs between the leading conditions.”
  - “A ‘yes’ answer would support …; a ‘no’ answer would make … less likely.”
- If probabilities are shown, they are presented as *model-based estimates* (not certainty).

**Safety / scope disclaimer:**
- This is an educational tool. It does not replace clinical judgment or medical advice.

In [20]:
from __future__ import annotations

from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple


def _base_of(token: str) -> str:
    """Extract evidence base from a token like 'E_66' or 'E_58=3'."""
    return token.split("=", 1)[0]


def neutralize_question_text(q: str) -> str:
    """
    Convert some 'you'-phrased questions into a more neutral clinical form.
    This is best-effort (heuristic), so it falls back to the original text if needed.
    """
    if not q:
        return ""

    q = q.strip()
    if not q.endswith("?"):
        q = q + "?"

    low = q.lower()

    # Handle a few common patterns first
    if low.startswith("are you experiencing "):
        return "Has there been " + q[len("Are you experiencing "):]
    if low.startswith("do you have "):
        return "Has there been " + q[len("Do you have "):]
    if low.startswith("have you had "):
        return "Has there been " + q[len("Have you had "):]
    if low.startswith("have you been "):
        return "Has there been " + q[len("Have you been "):]
    if low.startswith("have you "):
        return "Has there been " + q[len("Have you "):]

    # Soft fallback: keep wording but remove some direct 'you/your' where possible
    q = q.replace(" you ", " ").replace(" your ", " ")
    return q


def token_to_observation(token: str, evidences_map: Dict[str, Any]) -> str:
    """
    Convert an evidence token into a short, human-readable observation line.
    Example:
      'E_66'      -> 'Has there been shortness of breath ...: reported'
      'E_58=3'    -> '...: value=3'
    """
    base = _base_of(token)
    q = evidences_map.get(base, {}).get("question_en") or base
    q_neutral = neutralize_question_text(q).rstrip("?")

    if "=" in token:
        val = token.split("=", 1)[1]
        return f"{q_neutral}: value={val}"
    return f"{q_neutral}: reported"


def summarize_observations(
    tokens: Sequence[str],
    evidences_map: Dict[str, Any],
    *,
    max_lines: int = 6,
) -> List[str]:
    """Create a short bullet list of recorded findings so far."""
    lines: List[str] = []
    for t in tokens[:max_lines]:
        lines.append(f"- {token_to_observation(t, evidences_map)}")
    return lines


def explain_topk_diagnoses(
    topk: Sequence[Tuple[str, float]],
    tokens: Sequence[str],
    *,
    evidences_map: Dict[str, Any],
    conditions_map: Dict[str, Any],
    mini_manual: Optional[Dict[str, Any]] = None,
    retrieve_fn: Optional[Callable[[str, int], List[Dict[str, Any]]]] = None,
    k_retrieved: int = 2,
    max_supporting_findings: int = 3,
) -> str:
    """
    Plain-language explanation for 'Why these Top-k diagnoses now?'.

    mini_manual format (optional):
      mini_manual[condition_name] = {
        "summary": "1-2 sentence clinical summary ...",
        "sources": ["Short reference 1", "Short reference 2"]
      }

    retrieve_fn (optional) should return a list of dicts like:
      [{"text": "...", "meta": {...}}, ...]
    """
    mini_manual = mini_manual or {}
    observed_bases = {_base_of(t) for t in tokens}
    obs_lines = summarize_observations(tokens, evidences_map, max_lines=6)

    parts: List[str] = []
    parts.append("Based on the information recorded so far, these diagnoses are currently ranked highest.\n")
    if obs_lines:
        parts.append("**Recorded findings so far (abbreviated):**\n" + "\n".join(obs_lines) + "\n")

    parts.append("**Current leading diagnoses (Top‑k):**")
    for rank, (cond_name, p) in enumerate(topk, start=1):
        parts.append(f"\n**{rank}. {cond_name}** (estimated likelihood: {p:.1%})")

        # 1) Short clinical summary (prefer mini-manual)
        mm = mini_manual.get(cond_name, {})
        summary = (mm.get("summary") or "").strip()
        sources = mm.get("sources") or []

        if summary:
            parts.append(f"- *Brief description:* {summary}")
        else:
            icd10 = (conditions_map.get(cond_name, {}) or {}).get("icd10-id")
            if icd10:
                parts.append(f"- *Brief description:* Condition label in this dataset (ICD‑10: {icd10}).")
            else:
                parts.append("- *Brief description:* Condition label in this dataset.")

        # 2) Tie to recorded findings (via DDXPlus condition metadata)
        meta = conditions_map.get(cond_name, {}) or {}
        related = set((meta.get("symptoms") or {}).keys()) | set((meta.get("antecedents") or {}).keys())
        matched = [b for b in related if b in observed_bases][:max_supporting_findings]

        if matched:
            bullets: List[str] = []
            for b in matched:
                q = evidences_map.get(b, {}).get("question_en") or b
                bullets.append(neutralize_question_text(q).rstrip("?"))
            parts.append("- *Why it is on the list now:*")
            parts.extend([f"  - {x}" for x in bullets])
        else:
            parts.append("- *Why it is on the list now:* The current recorded findings are still compatible with this condition; more targeted findings may be needed to separate it from nearby alternatives.")

        # 3) Optional: show sources for the mini-manual
        if sources:
            parts.append("- *Sources:* " + "; ".join(str(s) for s in sources))

        # 4) Optional: add retrieved context snippets (kept short)
        if retrieve_fn is not None:
            hits = retrieve_fn(cond_name, k_retrieved) or []
            if hits:
                snippet = hits[0].get("text", "").strip()
                if snippet:
                    parts.append(f"- *Grounded note (local corpus):* {snippet}")

    parts.append("\n*Educational note:* This ranking is a model-based estimate from the recorded information so far; it is not a clinical diagnosis.")
    return "\n".join(parts)


def explain_next_question(
    next_base: str,
    topk: Sequence[Tuple[str, float]],
    *,
    evidences_map: Dict[str, Any],
    conditions_map: Dict[str, Any],
    mini_manual: Optional[Dict[str, Any]] = None,
    max_compare_conditions: int = 3,
) -> str:
    """
    Plain-language explanation for 'Why ask this next question?'

    This does NOT mention internal scoring details. It explains the clinical intuition:
    the answer tends to differ between leading possibilities.
    """
    mini_manual = mini_manual or {}
    q = evidences_map.get(next_base, {}).get("question_en") or next_base
    q_neutral = neutralize_question_text(q)

    leading = [c for c, _ in topk[:max_compare_conditions]]

    parts: List[str] = []
    parts.append(f"**Next question:** {q_neutral}\n")
    parts.append("**Why this question helps:**")
    parts.append("- This finding often differs between the leading possibilities, so the answer can quickly shift the ranking.")
    if leading:
        parts.append(f"- It is especially useful for separating: {', '.join(leading)}.")

    # Add a gentle, grounded hint from dataset metadata (not presented as a universal medical truth)
    mentioned_in = []
    for cond_name in leading:
        meta = conditions_map.get(cond_name, {}) or {}
        related = set((meta.get("symptoms") or {}).keys()) | set((meta.get("antecedents") or {}).keys())
        if next_base in related:
            mentioned_in.append(cond_name)
    if mentioned_in:
        parts.append(f"- In the DDXPlus condition mappings, this finding is listed for: {', '.join(mentioned_in)}.")

    parts.append("\n*Educational note:* If urgent or severe symptoms are present, clinical evaluation should not be delayed.")
    return "\n".join(parts)


## 4) End-to-end “assistant turn”: model → next question → grounded explanation

In this section, we connect the saved **token Logistic Regression** model and the **base-level question policy** to our **local retrieval corpus**.
The result is a reusable function that produces:
- Top‑k diagnoses (with probabilities)
- The next best question to ask
- Two user-facing explanations grounded in retrieved dataset context

**Important:** This is an educational prototype and does not provide medical advice.

In [21]:
# ============================================================
# 4.1 Validation check: artifacts already loaded (no re-loading)
# ============================================================

required_names = [
    # maps
    "evidences_map", "conditions_map",
    # preprocessors + feature spec
    "mlb_token", "ohe", "scaler", "label_encoder", "CAT_COLS", "NUM_COLS", "feature_spec",
    # model + policy
    "model_token", "p_e_given_d", "evidence_bases",
    # retriever
    "retr",
]

missing = [n for n in required_names if n not in globals()]
assert not missing, (
    "Missing required objects for the end-to-end loop:\n"
    f"{missing}\n\n"
    "Run the single 'Load artifacts (no retraining)' cell first (keep either 3.1 OR 4.1, not both)."
)

# Quick shape checks (helpful for catching subtle mismatches)
import numpy as np

assert len(getattr(mlb_token, "classes_", [])) == 972, "mlb_token vocab expected to be 972."
assert np.asarray(p_e_given_d).shape[1] == len(evidence_bases), "p_e_given_d columns must match evidence_bases."

print("✓ QC checks passed — ready for the end-to-end assistant loop.")

✓ QC checks passed — ready for the end-to-end assistant loop.


In [22]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


OUTPUTS_RAG_DIR  = PROJECT_ROOT / "outputs" / "rag"
RAG_INDEX_PATH = OUTPUTS_RAG_DIR / "tfidf_index.joblib"

def build_rag_corpus(
    evidences_map: Dict[str, Any],
    conditions_map: Dict[str, Any],
    *,
    mini_manual: Dict[str, str] | None = None,
):
    docs = []
    meta = []

    # Evidence documents
    for base, m in evidences_map.items():
        q = (m.get("question_en") or "").strip()
        dt = m.get("data_type")
        dv = m.get("default_value")
        docs.append(
            f"[EVIDENCE]\n"
            f"base: {base}\n"
            f"question: {q}\n"
            f"data_type: {dt}\n"
            f"default_value: {dv}\n"
        )
        meta.append({"type": "evidence", "base": base})

    # Condition documents
    for cond_name, m in conditions_map.items():
        icd = m.get("icd10-id", "")
        severity = m.get("severity", "")
        symptoms = list(m.get("symptoms", {}).keys())
        antecedents = list(m.get("antecedents", {}).keys())

        manual_text = ""
        if mini_manual and cond_name in mini_manual:
            manual_text = f"\nmini_manual: {mini_manual[cond_name].strip()}\n"

        docs.append(
            f"[CONDITION]\n"
            f"name: {cond_name}\n"
            f"icd10: {icd}\n"
            f"severity: {severity}\n"
            f"symptom_bases: {', '.join(symptoms)}\n"
            f"antecedent_bases: {', '.join(antecedents)}\n"
            f"{manual_text}"
        )
        meta.append({"type": "condition", "name": cond_name})

    return docs, meta

def build_tfidf_index(docs: list[str]):
    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    X = vec.fit_transform(docs)
    return {"vectorizer": vec, "doc_matrix": X, "docs": docs}

def rag_retrieve(query: str, index: Dict[str, Any], meta: list[dict], top_n: int = 5):
    vec = index["vectorizer"]
    X = index["doc_matrix"]
    docs = index["docs"]

    qv = vec.transform([query])
    sims = cosine_similarity(qv, X).ravel()
    top_idx = sims.argsort()[::-1][:top_n]

    hits = []
    for i in top_idx:
        hits.append(
            {"score": float(sims[i]), "meta": meta[i], "doc": docs[i]}
        )
    return hits

# If a mini-manual dict was created earlier, we reuse it; otherwise None.
mini_manual = globals().get("mini_manual", None)

if RAG_INDEX_PATH.exists():
    rag_index = joblib.load(RAG_INDEX_PATH)
    rag_meta = rag_index["meta"]
    rag_index = rag_index["index"]
    print(f"✓ Loaded TF-IDF index: {RAG_INDEX_PATH}")
else:
    rag_docs, rag_meta = build_rag_corpus(evidences_map, conditions_map, mini_manual=mini_manual)
    rag_index_obj = build_tfidf_index(rag_docs)
    joblib.dump({"index": rag_index_obj, "meta": rag_meta}, RAG_INDEX_PATH)
    rag_index = rag_index_obj
    print(f"✓ Built + saved TF-IDF index: {RAG_INDEX_PATH}")

print("✓ RAG corpus size:", len(rag_index["docs"]))

✓ Built + saved TF-IDF index: c:\Julia\Ironhack\Week20-24_Final_project\diagnostic-reasoning-assistant\outputs\rag\tfidf_index.joblib
✓ RAG corpus size: 272


In [23]:
from src.types import DiagnosisSession, parse_token

def _value_to_english(base: str, val: str, evidences_map: Dict[str, Any]) -> str:
    vm = evidences_map.get(base, {}).get("value_meaning", {}) or {}
    if val in vm and isinstance(vm[val], dict):
        return (vm[val].get("en") or val)
    return val

def token_to_readable(token: str, evidences_map: Dict[str, Any]) -> str:
    base, val = parse_token(token)
    q = (evidences_map.get(base, {}).get("question_en") or base).strip()

    if val is None:
        return f"{q} → yes"
    return f"{q} → {_value_to_english(base, val, evidences_map)}"

def session_known_positives(session: DiagnosisSession, evidences_map: Dict[str, Any], max_items: int = 8) -> list[str]:
    items = sorted(session.pos_items)
    readable = [token_to_readable(t, evidences_map) for t in items[:max_items]]
    if len(items) > max_items:
        readable.append(f"(+ {len(items) - max_items} more positive findings)")
    return readable

def condition_overlap_bases(condition_name: str, session: DiagnosisSession, conditions_map: Dict[str, Any]) -> Dict[str, list[str]]:
    cond = conditions_map.get(condition_name, {}) or {}
    symptom_bases = set((cond.get("symptoms", {}) or {}).keys())
    antecedent_bases = set((cond.get("antecedents", {}) or {}).keys())

    pos_bases = session.pos_bases
    return {
        "symptoms": sorted(pos_bases.intersection(symptom_bases)),
        "antecedents": sorted(pos_bases.intersection(antecedent_bases)),
    }

In [24]:
def explain_topk_diagnoses(
    session: DiagnosisSession,
    topk: list[tuple[str, float]],
    *,
    evidences_map: Dict[str, Any],
    conditions_map: Dict[str, Any],
    rag_index: Dict[str, Any],
    rag_meta: list[dict],
    mini_manual: Dict[str, str] | None = None,
    max_known: int = 6,
) -> str:
    known = session_known_positives(session, evidences_map, max_items=max_known)

    lines = []
    lines.append("Educational explanation (not medical advice).")
    if known:
        lines.append("What is known so far (positive findings):")
        for s in known:
            lines.append(f"  - {s}")
    else:
        lines.append("What is known so far: no positive findings recorded yet.")

    lines.append("")
    lines.append("Why these diagnoses are currently near the top:")
    for rank, (name, p) in enumerate(topk, start=1):
        overlap = condition_overlap_bases(name, session, conditions_map)
        ov_sym = overlap["symptoms"][:4]
        ov_ant = overlap["antecedents"][:3]

        # Retrieve one local snippet for grounding (condition doc / manual)
        hits = rag_retrieve(name, rag_index, rag_meta, top_n=1)
        snippet = hits[0]["doc"].splitlines()[0:6] if hits else []
        snippet_txt = " | ".join([x.strip() for x in snippet if x.strip()])

        manual = (mini_manual.get(name) if (mini_manual and name in mini_manual) else None)

        lines.append(f"{rank}) {name} (model probability ≈ {p:.3f})")
        if manual:
            lines.append(f"   Summary: {manual.strip()}")
        elif snippet_txt:
            lines.append(f"   Retrieved context: {snippet_txt}")

        if ov_sym or ov_ant:
            parts = []
            if ov_sym:
                parts.append(f"overlaps with listed symptoms: {', '.join(ov_sym)}")
            if ov_ant:
                parts.append(f"overlaps with listed risk/history items: {', '.join(ov_ant)}")
            lines.append("   Dataset grounding: " + "; ".join(parts) + ".")
        else:
            lines.append("   Dataset grounding: no direct overlap found with the recorded positives (still possible with limited evidence).")

    return "\n".join(lines)


def explain_next_question(
    session: DiagnosisSession,
    candidates: list[tuple[str, float, float, str]],
    topk: list[tuple[str, float]],
    *,
    evidences_map: Dict[str, Any],
    label_encoder,
    p_e_given_d: np.ndarray,
    evidence_index: Dict[str, int],
    rag_index: Dict[str, Any],
    rag_meta: list[dict],
    top_show: int = 3,
) -> str:
    if not candidates:
        return "No next-question candidates found (all evidence bases may already be asked)."

    base, score, p_yes, q_text = candidates[0]
    lines = []
    lines.append("Why we ask this next question (educational explanation):")
    lines.append(f"- Next question: {q_text.strip() if q_text else base}")
    lines.append(f"- This question was selected because it is expected to be especially helpful for telling the top diagnoses apart.")
    lines.append(f"- In the dataset, the chance of a 'yes' answer (given the current diagnosis probabilities) is ≈ {p_yes:.2f}.")

    # Show how it separates the top diagnoses using p(e=1|d) for top-k
    lines.append("")
    lines.append("How this question tends to differ across the leading diagnoses (dataset frequencies):")
    top_names = [n for n, _ in topk[:top_show]]
    j = evidence_index.get(base, None)
    if j is None:
        lines.append("- (This evidence base is not in the policy table index.)")
        return "\n".join(lines)

    for name in top_names:
        # label_encoder maps condition name <-> class index
        idx = int(label_encoder.transform([name])[0])
        pe = float(p_e_given_d[idx, j])
        lines.append(f"- {name}: ~{pe:.2f} of cases have this as 'yes'")

    # Retrieve evidence snippet (definition)
    hits = rag_retrieve(base, rag_index, rag_meta, top_n=1)
    if hits:
        snippet = hits[0]["doc"].splitlines()[0:6]
        snippet_txt = " | ".join([x.strip() for x in snippet if x.strip()])
        lines.append("")
        lines.append(f"Evidence definition (retrieved): {snippet_txt}")

    # Internal numeric score is kept but not called "information gain"
    lines.append("")
    lines.append(f"(Internal selection score ≈ {score:.4f}; higher means more expected usefulness.)")

    return "\n".join(lines)

In [25]:
from src.inference import predict_topk_from_session_token
from src.policy import choose_next_question_info_gain

def assistant_turn(
    session: DiagnosisSession,
    *,
    k: int = 5,
    top_n_questions: int = 8,
):
    # 1) Model inference (token encoding)
    topk, posterior = predict_topk_from_session_token(
        session=session,
        model=model_token,
        mlb_token=mlb_token,
        ohe=ohe,
        scaler=scaler,
        label_encoder=label_encoder,
        feature_spec=feature_spec,
        k=k,
    )

    # 2) Question policy (base-level)
    candidates = choose_next_question_info_gain(
        session=session,
        disease_proba=posterior,
        p_e_given_d=p_e_given_d,
        evidence_bases=evidence_bases,
        evidences_map=evidences_map,
        top_n=top_n_questions,
    )

    # 3) RAG-grounded explanations (user-facing)
    text_why_topk = explain_topk_diagnoses(
        session,
        topk,
        evidences_map=evidences_map,
        conditions_map=conditions_map,
        rag_index=rag_index,
        rag_meta=rag_meta,
        mini_manual=mini_manual,
    )
    text_why_next = explain_next_question(
        session,
        candidates,
        topk,
        evidences_map=evidences_map,
        label_encoder=label_encoder,
        p_e_given_d=p_e_given_d,
        evidence_index=evidence_index,
        rag_index=rag_index,
        rag_meta=rag_meta,
    )

    # 4) Package everything (nice for Streamlit later)
    next_question = candidates[0] if candidates else None  # (base, score, p_yes, q_text)

    return {
        "topk": topk,
        "posterior": posterior,
        "next_question": next_question,
        "candidates": candidates,
        "why_topk": text_why_topk,
        "why_next_question": text_why_next,
    }

In [27]:
# evidence_bases is the list of base evidence codes in the SAME order as columns in p_e_given_d
# p_e_given_d has shape (n_conditions, n_evidence_bases)
assert p_e_given_d.shape[1] == len(evidence_bases), "Policy table columns mismatch (p_e_given_d vs evidence_bases)"

# Map evidence base -> column index in p_e_given_d
evidence_index = {base: j for j, base in enumerate(evidence_bases)}

print("✓ Built evidence_index")
print("  n_bases:", len(evidence_index))
print("  sample:", list(evidence_index.items())[:5])

✓ Built evidence_index
  n_bases: 223
  sample: [('E_0', 0), ('E_1', 1), ('E_10', 2), ('E_100', 3), ('E_101', 4)]


In [28]:
# ============================================================
# assistant_turn: session -> posterior -> policy -> next Q
#      (self-contained: builds evidence_index if missing)


from typing import Dict, Any, List, Tuple, Optional
import numpy as np

from src.policy import update_posterior_with_answer  # uses evidence_index internally


def assistant_turn(
    session: DiagnosisSession,
    *,
    k: int = 5,
    top_n_questions: int = 6,
    beta_neg: float = 0.7,
) -> Dict[str, Any]:
    """
    One assistant turn:
      1) ML inference (token model) -> posterior over conditions
      2) Optional negative-answer update for the *policy posterior* (tempered Bayesian update)
      3) Info-gain policy -> ranked next question candidates
      4) Return a structured payload (RAG explanation added in later cells)
    """

    # ---- 1) ML inference (no retraining) ----
    topk, posterior = predict_topk_from_session_token(
        session,
        model_token,
        mlb_token=mlb_token,
        ohe=ohe,
        scaler=scaler,
        label_encoder=label_encoder,
        feature_spec=feature_spec,
        k=k,
    )
    posterior = np.asarray(posterior, dtype=float)

    # ---- 2) Policy posterior adjustment using negatives (optional) ----
    # Build evidence_index locally to avoid NameError if notebook cells are run out of order.
    local_evidence_index = {b: j for j, b in enumerate(evidence_bases)}
    assert p_e_given_d.shape[1] == len(evidence_bases)

    policy_posterior = posterior.copy()
    for base in sorted(session.neg_bases):
        policy_posterior = update_posterior_with_answer(
            policy_posterior,
            base,
            "no",
            p_e_given_d=p_e_given_d,
            evidence_index=local_evidence_index,
            beta=beta_neg,
        )

    # ---- 3) Next-question ranking (base-level IG) ----
    candidates = choose_next_question_info_gain(
        session=session,
        disease_proba=policy_posterior,
        p_e_given_d=p_e_given_d,
        evidence_bases=evidence_bases,
        evidences_map=evidences_map,
        top_n=top_n_questions,
    )

    chosen = candidates[0] if len(candidates) else None

    return {
        "topk": topk,                        # List[(condition_name, prob)]
        "posterior": posterior,              # np.ndarray, shape (n_conditions,)
        "policy_posterior": policy_posterior,
        "candidates": candidates,            # List[(base, ig, p_yes, question_text)]
        "chosen": chosen,                    # (base, ig, p_yes, question_text) or None
        "session": session,
    }

In [30]:
#  Smoke-demo: run one turn (should no longer NameError)
# Demo session (example positives; adjust freely)
session = DiagnosisSession(age=45.0, sex="F")
session.add_positive("E_66")
session.add_positive("E_58=3")
session.add_positive("E_58=7")

out = assistant_turn(session, k=5, top_n_questions=6)

print("Top-k conditions:")
for name, p in out["topk"]:
    print(f"  {name:35s}  {p:.4f}")

print("\nNext-question candidates (top):")
if out["chosen"] is None:
    print("  (no candidates left)")
else:
    base, ig, p_yes, q_text = out["chosen"]
    print(f"  base={base}  expected_helpfulness={ig:.4f}  P(yes)≈{p_yes:.3f}")
    print(f"  Q: {q_text}")

Top-k conditions:
  Guillain-Barré syndrome              0.9976
  Acute dystonic reactions             0.0018
  Myocarditis                          0.0002
  Myasthenia gravis                    0.0002
  Allergic sinusitis                   0.0001

Next-question candidates (top):
  base=E_62  expected_helpfulness=0.0087  P(yes)≈0.001
  Q: Do you regularly take stimulant drugs?
